In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from gsplat.rendering import rasterization

from nerfstudio.data.scene_box import SceneBox
from nerfstudio.cameras.cameras import Cameras, CameraType

from shadow_splat.model import ShadowSplatModel, ShadowSplatModelConfig, get_viewmat
from shadow_splat.util.general import generate_plane_points, generate_cylinder_points

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
config = ShadowSplatModelConfig()
scene_box = SceneBox(aabb=torch.tensor([[-1, -1, -1], [1, 1, 1]]))
model = ShadowSplatModel(config, scene_box, num_train_data=100).to(device)

# Initialize model points
plane_points = generate_plane_points(width=10.0, height=10.0, num_points=1000)
cylinder_points = generate_cylinder_points()
points = torch.cat([plane_points, cylinder_points], dim=0)
colors = 255 * torch.ones(points.shape[0], 3)
model.seed_points = (points, colors)
model.populate_modules()
model.training = False
model = model.to(device)

In [ ]:
torch.sigmoid(torch.tensor(1.0))

In [ ]:
torch.logit(torch.tensor(0.0))

In [ ]:
model.num_points

In [18]:
model.gauss_params["opacities"] = torch.logit(0.75 * torch.ones(model.num_points, 1))

In [ ]:
torch.sigmoid(model.opacities)

In [13]:
R = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
t = torch.tensor([[2.0], [0.0], [0.5]])

# Combine into camera_to_worlds matrix
camera_to_worlds = torch.cat([R, t], dim=1)
camera_to_worlds = camera_to_worlds.unsqueeze(0)  # Add batch dimension

# Define camera intrinsics
H = 360  # Height
W = 640  # Width
FOV = 60  # Field of view in degrees

# Convert FOV to radians and compute focal length
focal_length = W / (2 * np.tan(np.deg2rad(FOV / 2)))
fx = focal_length
fy = focal_length
cx = W / 2  # Principal point at image center
cy = H / 2

light_source = Cameras(
    camera_to_worlds=camera_to_worlds,
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy,
    width=W,
    height=H,
    camera_type=CameraType.PERSPECTIVE,
).to(device)

Unpack `update_light_source`

In [14]:
opacities_crop = model.opacities
means_crop = model.means
features_dc_crop = model.features_dc
features_rest_crop = model.features_rest
scales_crop = model.scales
quats_crop = model.quats

In [ ]:
optimized_light_to_world = light_source.camera_to_worlds

colors_crop = torch.cat((features_dc_crop[:, None, :], features_rest_crop), dim=1)

viewmat = get_viewmat(optimized_light_to_world)
K = light_source.get_intrinsics_matrices().cuda()
W, H = int(light_source.width.item()), int(light_source.height.item())

In [ ]:
depth, alphas, meta = rasterization(
    means=means_crop,
    quats=quats_crop,  # rasterization does normalization internally
    scales=torch.exp(scales_crop),
    opacities=torch.sigmoid(opacities_crop).squeeze(-1),
    colors=colors_crop,
    viewmats=viewmat,  # [1, 4, 4]
    Ks=K,  # [1, 3, 3]
    width=W,
    height=H,
    packed=True,
    near_plane=0.01,
    far_plane=1e10,
    render_mode="ED",
    sh_degree=0,
    sparse_grad=False,
    absgrad=False,
    camera_model="pinhole",
)

In [ ]:
plt.imshow(depth.squeeze().detach().cpu())

In [ ]:
alphas.shape

In [ ]:
meta.keys()

In [31]:
from gsplat.cuda._torch_impl import _world_to_cam

In [35]:
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()  # Ensure all prior work is done

# -------- Memory before --------
mem_before = torch.cuda.memory_allocated()

In [36]:
# Compute 3D camera space coordinates once and reuse them
w2c = torch.eye(4, device=light_source.camera_to_worlds[0].device)
w2c[:3] = light_source.camera_to_worlds[0, :3]
w2c = torch.linalg.inv(w2c)

means_camera_space = (w2c[:3, :3] @ means_crop.T).T + w2c[:3, 3][None]

In [ ]:
torch.cuda.synchronize()  # Make sure op is done

# -------- Memory after --------
mem_after = torch.cuda.memory_allocated()
peak_mem = torch.cuda.max_memory_allocated()

print(f"Memory used by operation: {(mem_after - mem_before) / 1024**2:.2f} MB")
print(f"Peak memory during operation: {(peak_mem - mem_before) / 1024**2:.2f} MB")